# Lekcja 10 - Agenci AI w produkcji

W tej lekcji poznasz **wzorce produkcyjne** dla agentów AI z użyciem Microsoft Agent Framework (Python). Omówimy:

- **Obserwowalność** — dodawanie pomiaru czasu i logowania interakcji agenta
- **Ocena** — użycie agenta ewaluatora do oceny jakości odpowiedzi
- **Zarządzanie kosztami** — strategie optymalizacji tokenów i wybór modelu

Scenariusz to **agent podróży**, który pomaga użytkownikom planować wycieczki, z warstwą monitorowania i oceny na górze.


## Konfiguracja


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Rozważania dotyczące produkcji

Przeniesienie agentów AI z prototypów do produkcji wymaga uważnej uwagi na trzy filary:

1. **Obserwowalność** — Musisz mieć wgląd w to, co agent robi, ile to trwa i jakich narzędzi używa. Bez śledzenia i logowania debugowanie problemów produkcyjnych jest niemal niemożliwe.

2. **Ocena** — Zautomatyzowane kontrole jakości zapewniają, że odpowiedzi agenta pozostają dokładne, kompletne i pomocne z upływem czasu. Agent oceniający może punktować odpowiedzi według zdefiniowanych kryteriów.

3. **Zarządzanie kosztami** — Wykorzystanie tokenów bezpośrednio wpływa na koszty. Strategie takie jak optymalizacja promptów, wybór modelu i buforowanie pomagają utrzymać wydatki pod kontrolą bez utraty jakości.


## Budowanie obserwowalnego agenta

Definiujemy narzędzia podróżne i otaczamy wywołanie agenta pomiarem czasu, aby móc monitorować opóźnienie. W produkcji zintegrowałbyś się z OpenTelemetry lub podobnym backendem do śledzenia.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Wzorce oceny

Częstym wzorcem produkcyjnym jest użycie drugiego agenta jako **ewaluatora**. Ewaluator ocenia odpowiedź głównego agenta na podstawie zdefiniowanych kryteriów, takich jak kompletność, dokładność i użyteczność.

To umożliwia:
- Automatyczne bramki jakości przed wysłaniem odpowiedzi do użytkowników
- Wykrywanie regresji, gdy zmieniają się podpowiedzi lub modele
- Ciągłe monitorowanie wydajności agenta w czasie


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## Strategie zarządzania kosztami

Kontrola kosztów jest kluczowa dla produkcyjnych agentów AI. Oto najważniejsze strategie:

| Strategia | Opis |
|---|---|
| **Optymalizacja promptu** | Utrzymuj instrukcje systemowe zwięzłe. Usuń zbędny kontekst, aby zmniejszyć liczbę tokenów wejściowych. |
    "| **Wybór modelu** | Używaj mniejszych, tańszych modeli (np. GPT-5-mini) do prostych zadań jak klasyfikacja czy ekstrakcja, a większe modele rezerwuj dla bardziej złożonego wnioskowania. |\n",
| **Buforowanie** | Buforuj wyniki narzędzi i częste zapytania, aby uniknąć powtarzających się wywołań API. |
| **Budżety tokenów** | Ustaw limity `max_tokens`, aby zapobiec niespodziewanie długim odpowiedziom. |
| **Grupowanie** | Grupuj wiele zapytań użytkowników w jedno wywołanie API tam, gdzie to możliwe. |

W praktyce dobrze sprawdza się podejście warstwowe: kieruj proste zapytania do szybkiego, taniego modelu, a tylko skomplikowane eskaluj do bardziej zaawansowanego.


## Podsumowanie

W tej lekcji nauczyłeś się, jak:

1. **Dodać obserwowalność** do interakcji agenta za pomocą pomiaru czasu i logowania, tworząc podstawy do śledzenia i monitorowania.
2. **Automatycznie oceniać odpowiedzi agenta** za pomocą agenta oceniającego, który punktuje kompletność, dokładność i pomocność.
3. **Zarządzać kosztami** poprzez optymalizację promptów, wybór modelu, buforowanie i budżety tokenów.

Te wzorce produkcyjne pomagają zapewnić, że Twoi agenci AI są niezawodni, mierzalni i opłacalni w skali.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Zastrzeżenie**:
Niniejszy dokument został przetłumaczony za pomocą usługi tłumaczenia AI [Co-op Translator](https://github.com/Azure/co-op-translator). Choć dążymy do dokładności, prosimy pamiętać, że automatyczne tłumaczenia mogą zawierać błędy lub niedokładności. Oryginalny dokument w jego języku źródłowym należy uznawać za autorytatywne źródło. W przypadku informacji krytycznych zalecane jest skorzystanie z profesjonalnego tłumaczenia wykonanego przez człowieka. Nie ponosimy odpowiedzialności za jakiekolwiek nieporozumienia lub błędne interpretacje wynikające z użycia tego tłumaczenia.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
